# Music Success Analysis - Part 2: Statistical Analysis

This notebook focuses on statistical analysis of the music streaming data, including basic statistics and correlation analysis.

## Loading Libraries and Data

First, let's import the necessary libraries and load our cleaned dataset.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import warnings

# Set plotting style and ignore warnings
sns.set_style('whitegrid')
warnings.filterwarnings('ignore')

# Display settings for better visualization
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Set random seed for reproducibility
np.random.seed(42)

In [ ]:
# Load the cleaned dataset from pickle file
try:
    df = pd.read_pickle('cleaned_music_data.pkl')
    print("Loaded cleaned data from pickle file.")
except FileNotFoundError:
    print("Cleaned data file not found. Please run the '1_Data_Loading_Cleaning.ipynb' notebook first.")
    # If pickle file not found, load from CSV as fallback
    file_path = r"C:\Users\Adilf\Downloads\Most Streamed Spotify Songs 2024.csv (1)\Most Streamed Spotify Songs 2024.csv"
    df = pd.read_csv(file_path, encoding='latin1')
    print("Loaded original data from CSV file as fallback.")

# Display the first few rows of the dataset
df.head()

## Basic Statistical Analysis

Let's calculate basic statistics for key metrics.

In [ ]:
# Calculate basic statistics for key metrics
numeric_stats = df[['Spotify Streams', 'YouTube Views', 'TikTok Views', 
                    'Spotify Popularity', 'Track Score']].describe()

print("Basic statistics for key metrics:")
numeric_stats

In [ ]:
# Create a box plot to visualize the distribution of key metrics
# First, normalize the data for better visualization
metrics = ['Spotify Streams', 'YouTube Views', 'TikTok Views', 'Spotify Popularity', 'Track Score']
normalized_df = pd.DataFrame()

for metric in metrics:
    normalized_df[metric] = (df[metric] - df[metric].min()) / (df[metric].max() - df[metric].min())

# Melt the dataframe for easier plotting
melted_df = pd.melt(normalized_df, var_name='Metric', value_name='Normalized Value')

# Create the box plot
plt.figure(figsize=(14, 8))
sns.boxplot(x='Metric', y='Normalized Value', data=melted_df, palette='viridis')
plt.title('Distribution of Key Metrics (Normalized)', fontsize=16)
plt.xlabel('Metric', fontsize=12)
plt.ylabel('Normalized Value', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## Correlation Analysis

Let's compute correlations between different platforms and metrics.

In [ ]:
# Compute correlations between platforms
correlation_columns = [
    'Spotify Streams', 'YouTube Views', 'TikTok Views', 'TikTok Likes',
    'Spotify Playlist Count', 'Spotify Popularity', 'Track Score'
]

# Calculate correlation matrix
correlation_matrix = df[correlation_columns].corr()
correlation_matrix

In [ ]:
# Create a correlation heatmap
plt.figure(figsize=(12, 10))
mask = np.triu(correlation_matrix)
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt='.2f',
            linewidths=0.5, mask=mask)
plt.title('Correlation Between Different Platforms', fontsize=16)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Hypothesis Testing

Let's perform some hypothesis tests to answer key research questions.

In [ ]:
# Hypothesis test: Do explicit tracks perform differently than non-explicit tracks?
# Compare performance metrics between explicit and non-explicit tracks
performance_metrics = ['Spotify Streams', 'YouTube Views', 'TikTok Views', 'Track Score']

# Create a DataFrame to store the comparison
explicit_comparison = pd.DataFrame()

for metric in performance_metrics:
    explicit_tracks_metric = df[df['Explicit Track'] == 1][metric].dropna()
    non_explicit_tracks_metric = df[df['Explicit Track'] == 0][metric].dropna()
    
    # Perform t-test
    t_stat, p_value = stats.ttest_ind(explicit_tracks_metric, non_explicit_tracks_metric, equal_var=False)
    
    explicit_comparison = explicit_comparison.append({
        'Metric': metric,
        'Explicit Mean': explicit_tracks_metric.mean(),
        'Non-Explicit Mean': non_explicit_tracks_metric.mean(),
        'Difference (%)': (explicit_tracks_metric.mean() - non_explicit_tracks_metric.mean()) / non_explicit_tracks_metric.mean() * 100,
        'T-Statistic': t_stat,
        'P-Value': p_value,
        'Significant': p_value < 0.05
    }, ignore_index=True)

# Display the comparison
print("Comparison of Explicit vs Non-Explicit Tracks:")
explicit_comparison

In [ ]:
# Hypothesis test: Does playlist inclusion significantly impact streaming numbers?
# Create bins for playlist count
df['Playlist_Bins'] = pd.cut(df['Spotify Playlist Count'], 
                            bins=[0, 10000, 20000, 30000, 40000, 50000, 100000],
                            labels=['0-10K', '10K-20K', '20K-30K', '30K-40K', '40K-50K', '50K+'])

# Perform ANOVA to test if there are significant differences between playlist bins
playlist_groups = [df[df['Playlist_Bins'] == bin]['Spotify Streams'].dropna() for bin in df['Playlist_Bins'].unique()]
f_stat, p_value = stats.f_oneway(*playlist_groups)

print(f"ANOVA Test for Playlist Impact on Streaming Numbers:")
print(f"F-statistic: {f_stat:.4f}")
print(f"P-value: {p_value:.4f}")
print(f"Significant difference: {p_value < 0.05}")

# Calculate correlation between playlist count and streams
playlist_correlation = df['Spotify Playlist Count'].corr(df['Spotify Streams'])
print(f"\nCorrelation between Playlist Count and Streams: {playlist_correlation:.4f}")

In [ ]:
# Hypothesis test: Are there optimal times of year for music releases?
# Perform ANOVA to test if there are significant differences between release months
month_groups = [df[df['Release Month'] == month]['Spotify Streams'].dropna() for month in range(1, 13)]
f_stat, p_value = stats.f_oneway(*month_groups)

print(f"ANOVA Test for Release Month Impact on Streaming Numbers:")
print(f"F-statistic: {f_stat:.4f}")
print(f"P-value: {p_value:.4f}")
print(f"Significant difference: {p_value < 0.05}")

# Group by release month and calculate average streams
monthly_data = df.groupby('Release Month')['Spotify Streams'].agg(['count', 'mean'])
monthly_data.columns = ['Number of Releases', 'Average Streams']
monthly_data = monthly_data.reset_index()

# Add month names for better readability
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly_data['Month Name'] = monthly_data['Release Month'].apply(lambda x: month_names[x-1])

# Sort by average streams to find optimal months
optimal_months = monthly_data.sort_values('Average Streams', ascending=False)
print("\nMonths Ranked by Average Spotify Streams:")
print(optimal_months[['Month Name', 'Average Streams', 'Number of Releases']])